step 1 create dataset

In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
np.random.seed(42)
n = 200
locations = ['Urban', 'Suburban', 'Rural']
df = pd.DataFrame({
 'area_sqft': np.random.randint(400, 2000, n),
 'num_rooms': np.random.randint(1, 7, n),
 'location': np.random.choice(locations, n),
 'age_years': np.random.randint(0, 40, n),
})

loc_map = {'Urban': 1.5, 'Suburban': 1.0, 'Rural': 0.7}
noise = np.random.normal(0, 20000, n)
df['price'] = (
 df['area_sqft'] * 150 +
 df['num_rooms'] * 25000 -
 df['age_years'] * 2000 +
 df['location'].map(loc_map) * 50000 +
 noise
).astype(int)
print(df.head(8).to_string(index=False))
print(f"\nDataset shape : {df.shape}")
print(df.describe()[['area_sqft','num_rooms','age_years','price']].round(0))

 area_sqft  num_rooms location  age_years  price
      1526          3    Rural         11 328774
      1859          6 Suburban         12 479106
      1260          1    Rural         22 209062
      1694          4 Suburban         24 357091
      1530          5    Rural         34 353711
      1495          1    Rural         29 251729
      1444          3 Suburban         16 292228
       521          6 Suburban         19 242478

Dataset shape : (200, 5)
       area_sqft  num_rooms  age_years     price
count      200.0      200.0      200.0     200.0
mean      1257.0        3.0       19.0  291785.0
std        464.0        2.0       11.0   91044.0
min        413.0        1.0        0.0   67451.0
25%        882.0        2.0       10.0  228315.0
50%       1266.0        3.0       18.0  297670.0
75%       1656.0        5.0       29.0  344523.0
max       1998.0        6.0       39.0  518640.0


step 2 Encode & prepare features

In [2]:
le = LabelEncoder()
df['location_enc'] = le.fit_transform(df['location'])

features = ['area_sqft', 'num_rooms', 'age_years', 'location_enc']
X = df[features]
y = df['price']

print(df[['location', 'location_enc']].drop_duplicates().sort_values('location_enc'))
print(f"\nFeature matrix shape : {X.shape}")
print(f"Target vector shape : {y.shape}")

   location  location_enc
0     Rural             0
6  Suburban             1
1     Urban             2

Feature matrix shape : (200, 4)
Target vector shape : (200,)


step 3 Train Test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Testing samples : {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")
print(f"Feature names : {features}")

Training samples : 160 (80%)
Testing samples : 40 (20%)
Feature names : ['area_sqft', 'num_rooms', 'age_years', 'location_enc']


step 4 Train the Model

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Learned Coefficients:")
print("-" * 45)

for feat, coef in zip(features, model.coef_):
 direction = "↑" if coef > 0 else "↓"
 print(f" {feat:15s}: Rs {coef:>10,.0f} {direction}")

print(f" {'intercept':15s}: Rs {model.intercept_:>10,.0f}")
print("-" * 45)

Learned Coefficients:
---------------------------------------------
 area_sqft      : Rs        151 ↑
 num_rooms      : Rs     23,169 ↑
 age_years      : Rs     -2,262 ↓
 location_enc   : Rs     18,114 ↑
 intercept      : Rs     46,789
---------------------------------------------


step 5 Evaluate

In [5]:
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 40)
print(" MODEL EVALUATION REPORT")
print("=" * 40)
print(f"R² Score : {r2:.4f} {'✓ Excellent' if r2>0.9 else '✓ Good' if r2>0.8 else '■ Needs work'}")
print(f"MAE : Rs {mae:>10,.0f}")
print(f"RMSE : Rs {rmse:>10,.0f}")
print("=" * 40)

print("\nSample Predictions:")
print(f"{'Actual':>10} | {'Predicted':>10} | {'Error':>10}")
print("-" * 36)
for a, p in zip(y_test[:8], y_pred[:8]):
 print(f"{a:>10,} | {p:>10,.0f} | {p-a:>+10,.0f}")

 MODEL EVALUATION REPORT
R² Score : 0.9872 ✓ Excellent
MAE : Rs     13,439
RMSE : Rs     16,959

Sample Predictions:
    Actual |  Predicted |      Error
------------------------------------
   489,899 |    478,320 |    -11,579
   601,329 |    572,877 |    -28,452
   623,788 |    650,489 |    +26,701
   291,458 |    271,086 |    -20,372
   390,040 |    344,431 |    -45,609
   123,884 |    134,545 |    +10,661
   484,442 |    443,295 |    -41,147
   562,823 |    556,350 |     -6,473


step 6 predict on new house

In [12]:
new_houses = pd.DataFrame({
 'area_sqft': [1000, 3000, 1800, 3200],
 'num_rooms': [ 1, 6 ,5 ,4],
 'age_years': [ 25, 2 ,8, 9],
 'location_enc':[ 0, 2 ,4 ,3],
 'location_lbl':['Rural','Urban','Suburban','urban'],
})
predictions = model.predict(new_houses[features])
print("\n HOUSE PRICE PREDICTIONS")
print("=" * 60)
for i, (_, row) in enumerate(new_houses.iterrows()):
 print(f" House {i+1}: {row.area_sqft} sqft | "
 f"{row.num_rooms} rooms | "
 f"{row.age_years}yr old | "
 f"{row.location_lbl}")
 print(f" Estimated Price -> Rs {predictions[i]:>10,.0f}")
 print()


 HOUSE PRICE PREDICTIONS
 House 1: 1000 sqft | 1 rooms | 25yr old | Rural
 Estimated Price -> Rs    163,989

 House 2: 3000 sqft | 6 rooms | 2yr old | Urban
 Estimated Price -> Rs    669,260

 House 3: 1800 sqft | 5 rooms | 8yr old | Suburban
 Estimated Price -> Rs    488,044

 House 4: 3200 sqft | 4 rooms | 9yr old | urban
 Estimated Price -> Rs    655,318

